# Análise dos resultados do benchmark no Arduino Nano 33 BLE Sense Rev2

Lê os CSVs gerados pelo `run_benchmark.py` (pasta `resultados/`) e produz tabelas, gráficos e um resumo automático de insights sobre **latência, memória, acurácia e trade-offs** entre FP32, INT16 (16x8) e INT8.

**Como usar:** ajuste `RESULTS_DIR` e `EXPORT_DIR` na primeira célula e rode tudo. Figuras e tabelas ficam em `resultados/figuras/` e `resultados/analise/`.

Energia ainda não entra na análise. A última seção tem uma célula pronta para calcular a energia por inferência e o EDP quando você tiver as medidas.

In [ ]:
import importlib, subprocess, sys
for pkg in ("numpy", "pandas", "matplotlib", "scipy"):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown

# ============================ CONFIGURAÇÃO ============================
RESULTS_DIR = Path("resultados")      # pasta de saída do run_benchmark.py
EXPORT_DIR = Path("arduino_export")   # opcional: nomes das classes e referência do PC (meta.json)
FS_HZ = 12000                         # taxa de amostragem do CWRU (12k)
OVERLAP = 0.5                         # sobreposição das janelas usada no notebook de treino
# =====================================================================

FIG_DIR = RESULTS_DIR / "figuras"
TAB_DIR = RESULTS_DIR / "analise"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

ORDEM = ["FP32", "INT16", "INT8"]
CORES = {"FP32": "#1f77b4", "INT16": "#ff7f0e", "INT8": "#2ca02c"}

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})


def salvar_fig(fig, nome):
    fig.savefig(FIG_DIR / f"{nome}.png", dpi=200, bbox_inches="tight")


def md(texto):
    display(Markdown(texto))

## 1. Carga dos dados

In [ ]:
def ler(nome, obrigatorio=True):
    p = RESULTS_DIR / nome
    if not p.exists():
        if obrigatorio:
            raise FileNotFoundError(f"{p.resolve()} não encontrado. Rode o run_benchmark.py ou ajuste RESULTS_DIR.")
        print(f"(aviso) {nome} não encontrado: as seções que dependem dele serão puladas.")
        return None
    return pd.read_csv(p)


def json_ou_vazio(p):
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}


sessoes = ler("resumo_por_sessao.csv")
lat = ler("latencia_bruta.csv", obrigatorio=False)
acc = ler("acuracia_na_placa.csv", obrigatorio=False)
meta_sessao = json_ou_vazio(RESULTS_DIR / "sessao_meta.json")
meta = json_ou_vazio(EXPORT_DIR / "meta.json")

ok = sessoes[sessoes["status"] == "ok"].copy()
VAR_OK = [v for v in ORDEM if v in set(ok["variante"])]
VAR_TODAS = [v for v in ORDEM if v in set(sessoes["variante"])]
nao_exec = sessoes[sessoes["status"] != "ok"].drop_duplicates("variante", keep="last").set_index("variante")
primeira = ok.sort_values("sessao").groupby("variante").first()   # 1º valor não nulo de cada coluna

if lat is not None:
    lat = lat.sort_values(["variante", "sessao", "run"]).copy()
    lat["ms"] = lat["invoke_us"] / 1000.0

CLASSES = list(meta.get("classes", []))
if acc is not None and len(acc):
    k_max = int(max(acc["y_true"].max(), acc["y_pred"].max())) + 1
    CLASSES += [f"classe {i}" for i in range(len(CLASSES), k_max)]
K = len(CLASSES)
WINDOW_LEN = int(meta.get("window_len", 2048))

print(f"Sessões: {sessoes['sessao'].nunique()} | executadas: {VAR_OK} | não executadas: {list(nao_exec.index)}")
if meta_sessao:
    print("Data da campanha:", meta_sessao.get("data"))
    for campo, filtro in (("cores", "mbed"), ("bibliotecas", "TensorFlow")):
        for linha in str(meta_sessao.get(campo, "")).splitlines():
            if filtro.lower() in linha.lower():
                print(" ", " ".join(linha.split()))

## 2. Panorama

In [ ]:
def lat_stats(x):
    q1, q3 = np.percentile(x, [25, 75])
    dp = x.std(ddof=1) if len(x) > 1 else 0.0
    return {"n": len(x), "média_ms": x.mean(), "dp_ms": dp, "cv_%": 100 * dp / x.mean(),
            "mediana_ms": np.median(x), "iqr_ms": q3 - q1, "p95_ms": np.percentile(x, 95),
            "p99_ms": np.percentile(x, 99), "mín_ms": x.min(), "máx_ms": x.max()}


if lat is not None:
    estat = pd.DataFrame.from_dict({v: lat_stats(g["ms"].to_numpy()) for v, g in lat.groupby("variante")},
                                   orient="index").reindex([v for v in VAR_OK if v in set(lat["variante"])])
else:
    estat = pd.DataFrame()

STATUS_TXT = {"no_fit_ram": "não cabe na RAM", "no_fit_arena": "arena insuficiente", "no_fit": "nenhum arena serviu",
              "boot_failed": "placa não iniciou", "alloc_failed": "alocação falhou"}

linhas = []
for v in VAR_TODAS:
    if v in VAR_OK:
        r = primeira.loc[v]
        linhas.append({"variante": v, "situação": "ok",
                       "latência_ms": estat.loc[v, "média_ms"] if v in estat.index else np.nan,
                       "RAM_%": r.get("ram_pct"), "Flash_%": r.get("flash_pct"),
                       "arena_KB": r.get("arena_used_bytes", np.nan) / 1024,
                       "acurácia": r.get("acc_placa"), "F1_macro": r.get("f1_placa")})
    else:
        linhas.append({"variante": v, "situação": STATUS_TXT.get(nao_exec.loc[v, "status"], nao_exec.loc[v, "status"])})
panorama = pd.DataFrame(linhas).set_index("variante")
display(panorama.round(3))
panorama.to_csv(TAB_DIR / "panorama.csv", encoding="utf-8-sig")

## 3. Latência

In [ ]:
if lat is None or estat.empty:
    print("Sem dados de latência.")
else:
    md("**Estatísticas por versão** (todas as execuções de todas as sessões, em ms)")
    display(estat.round(3))
    if sessoes["sessao"].nunique() > 1:
        md("**Média por sessão (ms)** — a ordem das versões muda a cada sessão")
        display(lat.pivot_table(index="sessao", columns="variante", values="ms", aggfunc="mean")[list(estat.index)].round(3))
    estat.to_csv(TAB_DIR / "latencia_estatisticas.csv", encoding="utf-8-sig")

In [ ]:
if lat is not None and not estat.empty:
    vs = list(estat.index)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    dados = [lat.loc[lat["variante"] == v, "ms"].to_numpy() for v in vs]
    bp = ax[0].boxplot(dados, patch_artist=True, showfliers=True, flierprops={"markersize": 3, "alpha": 0.5})
    ax[0].set_xticks(range(1, len(vs) + 1)); ax[0].set_xticklabels(vs)
    for patch, v in zip(bp["boxes"], vs):
        patch.set_facecolor(CORES[v]); patch.set_alpha(0.6)
    ax[0].set_ylabel("latência do Invoke() (ms)"); ax[0].set_title("Distribuição por versão")

    ax[1].bar(vs, estat["média_ms"], yerr=estat["dp_ms"], color=[CORES[v] for v in vs], alpha=0.75, capsize=5)
    for i, v in enumerate(vs):
        ax[1].text(i, estat.loc[v, "média_ms"], f"{estat.loc[v, 'média_ms']:.2f}", ha="center", va="bottom", fontsize=9)
    ax[1].set_ylabel("ms"); ax[1].set_title("Média ± desvio padrão")
    fig.tight_layout(); salvar_fig(fig, "latencia_distribuicao"); plt.show()

    fig, axs = plt.subplots(1, len(vs), figsize=(4.2 * len(vs), 3.4), squeeze=False)
    for a, v in zip(axs[0], vs):
        x = lat.loc[lat["variante"] == v, "ms"].to_numpy()
        a.hist(x, bins=40, color=CORES[v], alpha=0.75)
        a.axvline(np.median(x), color="k", lw=1.2, label=f"mediana {np.median(x):.2f}")
        a.axvline(np.percentile(x, 95), color="r", lw=1.2, ls="--", label=f"p95 {np.percentile(x, 95):.2f}")
        a.set_title(v); a.set_xlabel("ms"); a.legend(fontsize=8)
    fig.tight_layout(); salvar_fig(fig, "latencia_histogramas"); plt.show()

### Qualidade da medição

Antes de comparar versões, vale conferir se as medidas são confiáveis: **picos** (interrupções de USB/RTOS), **deriva** (aquecimento, tensão), **autocorrelação** entre execuções consecutivas e **diferença entre sessões**.

In [ ]:
if lat is not None and not estat.empty:
    linhas = []
    for v in estat.index:
        g = lat[lat["variante"] == v]
        x = g["ms"].to_numpy()
        med = np.median(x)
        mad = np.median(np.abs(x - med)) * 1.4826
        limite = med + 5 * (mad if mad > 0 else x.std())
        n_out = int((x > limite).sum())
        # deriva: inclinação da reta (latência x execução) dentro de cada sessão, em % da média por 100 execuções
        deriva, p_deriva = [], []
        for _, gs in g.groupby("sessao"):
            if len(gs) > 10:
                reg = stats.linregress(gs["run"], gs["ms"])
                deriva.append(100 * reg.slope * 100 / gs["ms"].mean()); p_deriva.append(reg.pvalue)
        pior = int(np.argmax(np.abs(deriva))) if deriva else 0
        ac1 = float(np.corrcoef(x[:-1], x[1:])[0, 1]) if len(x) > 2 else np.nan
        medias = g.groupby("sessao")["ms"].mean()
        p_sessao = np.nan
        if g["sessao"].nunique() > 1:
            p_sessao = stats.kruskal(*[gs["ms"].to_numpy() for _, gs in g.groupby("sessao")]).pvalue
        linhas.append({"variante": v, "picos_>5MAD_%": 100 * n_out / len(x), "máx/mediana": x.max() / med,
                       "deriva_%_por_100_exec": deriva[pior] if deriva else np.nan,
                       "p_deriva": p_deriva[pior] if p_deriva else np.nan,
                       "autocorr_lag1": ac1,
                       "variação_entre_sessões_%": 100 * (medias.max() - medias.min()) / medias.mean() if len(medias) > 1 else np.nan,
                       "p_sessões_kruskal": p_sessao})
    qual = pd.DataFrame(linhas).set_index("variante")
    display(qual.round(4))
    qual.to_csv(TAB_DIR / "latencia_qualidade.csv", encoding="utf-8-sig")

    fig, axs = plt.subplots(1, len(estat), figsize=(4.6 * len(estat), 3.4), squeeze=False)
    for a, v in zip(axs[0], estat.index):
        for s, gs in lat[lat["variante"] == v].groupby("sessao"):
            a.plot(gs["run"], gs["ms"], lw=0.5, alpha=0.35, color=CORES[v])
            a.plot(gs["run"], gs["ms"].rolling(25, center=True, min_periods=1).median(), lw=1.6, label=f"sessão {s}")
        a.set_title(v); a.set_xlabel("execução"); a.set_ylabel("ms"); a.legend(fontsize=8)
    fig.suptitle("Latência ao longo das execuções (linha grossa: mediana móvel de 25)", y=1.02)
    fig.tight_layout(); salvar_fig(fig, "latencia_serie_temporal"); plt.show()
else:
    qual = pd.DataFrame()

In [ ]:
# A latência foi medida repetindo UMA janela. Isso só vale se o tempo não depender do conteúdo do sinal:
# a passada de acurácia mede o Invoke() de cada janela de teste, com classes diferentes.
indep = pd.DataFrame()
if acc is not None and lat is not None and not estat.empty:
    # Se as janelas foram processadas em ordem (classes em blocos), tempo e classe se confundem e o teste não vale.
    ordenada = bool(acc.groupby("variante")["janela"].apply(lambda s: s.is_monotonic_increasing).all())
    linhas = []
    for v in estat.index:
        sub = acc[acc["variante"] == v]
        if sub.empty:
            continue
        p_classe = np.nan
        if not ordenada:
            grupos = [g["invoke_us"].to_numpy() / 1000 for _, g in sub.groupby("y_true") if len(g) > 1]
            p_classe = stats.kruskal(*grupos).pvalue if len(grupos) > 1 else np.nan
        linhas.append({"variante": v, "média_janelas_ms": sub["invoke_us"].mean() / 1000,
                       "média_mesma_janela_ms": estat.loc[v, "média_ms"],
                       "razão": sub["invoke_us"].mean() / 1000 / estat.loc[v, "média_ms"],
                       "p_diferença_entre_classes": p_classe})
    if linhas:
        indep = pd.DataFrame(linhas).set_index("variante")
        display(indep.round(4))
        if ordenada:
            md("**Teste entre classes não aplicado:** nesta campanha as janelas foram processadas em ordem (classes em blocos), então o efeito do tempo "
               "(aquecimento, carga do USB) se confunde com o da classe. O `run_benchmark.py` atual embaralha a ordem; rode de novo para ter este teste.")
        else:
            md("`razão` acima de 1 é esperado: na passada de acurácia cada janela roda uma vez só, logo depois da comunicação serial, "
               "sem aquecimento. O que importa é `p_diferença_entre_classes`: valores altos indicam que o conteúdo do sinal não muda o tempo.")

### Comparação entre versões

Razão entre as médias de latência, com intervalo de confiança de 95% por *block bootstrap* (blocos de 20 execuções consecutivas, para respeitar a autocorrelação). Razão > 1 significa que a segunda versão é mais rápida.

In [ ]:
def bootstrap_bloco(x, y, stat=np.mean, bloco=20, B=2000, seed=0):
    """IC 95% de stat(x)/stat(y), reamostrando blocos consecutivos."""
    rng = np.random.default_rng(seed)

    def reamostra(v):
        n = len(v); b = min(bloco, n); nb = math.ceil(n / b)
        ini = rng.integers(0, n - b + 1, size=(B, nb))
        idx = (ini[:, :, None] + np.arange(b)).reshape(B, -1)[:, :n]
        return stat(v[idx], axis=1)
    return np.percentile(reamostra(x) / reamostra(y), [2.5, 97.5])


comp = pd.DataFrame()
if lat is not None and len(estat) > 1:
    linhas = []
    for i, a in enumerate(estat.index):
        for b in estat.index[i + 1:]:
            xa = lat.loc[lat["variante"] == a, "ms"].to_numpy(); xb = lat.loc[lat["variante"] == b, "ms"].to_numpy()
            lo, hi = bootstrap_bloco(xa, xb)
            linhas.append({"comparação": f"{a} → {b}", "razão_médias": xa.mean() / xb.mean(), "IC95_inf": lo, "IC95_sup": hi,
                           "razão_medianas": np.median(xa) / np.median(xb),
                           "variação_%": 100 * (xb.mean() / xa.mean() - 1)})
    comp = pd.DataFrame(linhas).set_index("comparação")
    display(comp.round(3))
    comp.to_csv(TAB_DIR / "latencia_comparacoes.csv", encoding="utf-8-sig")
else:
    print("Comparação precisa de pelo menos duas versões executadas.")

### Orçamento de tempo real

Uma janela tem `WINDOW_LEN / FS` segundos de sinal; com sobreposição, uma nova janela fecha a cada `hop`. Se a inferência demora mais que o `hop`, a placa não acompanha o sensor.

In [ ]:
rt = pd.DataFrame()
if not estat.empty:
    janela_ms = 1000 * WINDOW_LEN / FS_HZ
    hop_ms = janela_ms * (1 - OVERLAP)
    rt = pd.DataFrame({"latência_ms": estat["média_ms"],
                       "uso_do_intervalo_%": 100 * estat["média_ms"] / hop_ms,
                       "folga_ms": hop_ms - estat["média_ms"],
                       "janelas_por_s": 1000 / estat["média_ms"]})
    print(f"Janela: {WINDOW_LEN} amostras a {FS_HZ} Hz = {janela_ms:.1f} ms de sinal | hop com {OVERLAP:.0%} de sobreposição = {hop_ms:.1f} ms")
    display(rt.round(3))
    rt.to_csv(TAB_DIR / "tempo_real.csv", encoding="utf-8-sig")

    fig, ax = plt.subplots(figsize=(6, 3.4))
    ax.barh(rt.index, rt["uso_do_intervalo_%"], color=[CORES[v] for v in rt.index], alpha=0.8)
    ax.axvline(100, color="r", ls="--", lw=1.2, label="limite de tempo real")
    for i, v in enumerate(rt.index):
        ax.text(rt.loc[v, "uso_do_intervalo_%"], i, f" {rt.loc[v, 'uso_do_intervalo_%']:.1f}%", va="center")
    ax.set_xlabel("% do intervalo entre janelas ocupado pela inferência"); ax.legend()
    fig.tight_layout(); salvar_fig(fig, "tempo_real"); plt.show()

## 4. Memória

A RAM estática que o compilador informa inclui o Mbed OS, o TFLM, os buffers do sketch de teste (`harness`) e o arena de tensores. Separando essas partes, dá para ver **o que muda entre as versões**: em princípio, só o arena e o tamanho do modelo.

In [ ]:
mem = pd.DataFrame()
if VAR_OK:
    cols = ["flash_bytes", "flash_max_bytes", "ram_bytes", "ram_max_bytes", "arena_kb", "arena_used_bytes", "harness_bytes", "tflite_bytes"]
    mem = primeira.loc[VAR_OK, [c for c in cols if c in primeira.columns]].copy()
    mem["arena_alocado_bytes"] = mem["arena_kb"] * 1024
    mem["ram_base_bytes"] = mem["ram_bytes"] - mem["arena_alocado_bytes"] - mem["harness_bytes"]      # Mbed OS + TFLM + demais globais
    mem["flash_runtime_bytes"] = mem["flash_bytes"] - mem["tflite_bytes"]                            # Mbed OS + TFLM + sketch
    mem["ram_livre_KB"] = (mem["ram_max_bytes"] - mem["ram_bytes"]) / 1024
    mem["flash_livre_KB"] = (mem["flash_max_bytes"] - mem["flash_bytes"]) / 1024
    tab = pd.DataFrame({
        "RAM_%": 100 * mem["ram_bytes"] / mem["ram_max_bytes"],
        "RAM_livre_KB": mem["ram_livre_KB"],
        "arena_usado_KB": mem["arena_used_bytes"] / 1024,
        "RAM_base_KB": mem["ram_base_bytes"] / 1024,
        "Flash_%": 100 * mem["flash_bytes"] / mem["flash_max_bytes"],
        "Flash_livre_KB": mem["flash_livre_KB"],
        "modelo_KB": mem["tflite_bytes"] / 1024,
        "Flash_runtime_KB": mem["flash_runtime_bytes"] / 1024})
    display(tab.round(2))
    tab.to_csv(TAB_DIR / "memoria.csv", encoding="utf-8-sig")

    if len(VAR_OK) > 1:
        base = mem["ram_base_bytes"] / 1024
        md(f"A RAM base (tudo que **não** é arena nem buffer de teste) varia {base.max() - base.min():.2f} KB entre as versões "
           f"({base.min():.1f} a {base.max():.1f} KB). Se for próximo de zero, a diferença de RAM vem só do arena.")

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    y = np.arange(len(VAR_OK))
    partes = [("ram_base_bytes", "Mbed OS + TFLM + globais", "#999999"), ("harness_bytes", "buffers do teste", "#cccccc"),
              ("arena_alocado_bytes", "arena de tensores", "#4c72b0")]
    esq = np.zeros(len(VAR_OK))
    for col, nome, cor in partes:
        v = mem[col].to_numpy() / 1024
        ax[0].barh(y, v, left=esq, color=cor, label=nome); esq += v
    ax[0].axvline(mem["ram_max_bytes"].iloc[0] / 1024, color="r", ls="--", label="capacidade")
    ax[0].set_yticks(y); ax[0].set_yticklabels(VAR_OK); ax[0].set_xlabel("KB"); ax[0].set_title("RAM estática")
    ax[0].legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False)

    esq = np.zeros(len(VAR_OK))
    for col, nome, cor in (("flash_runtime_bytes", "Mbed OS + TFLM + sketch", "#999999"), ("tflite_bytes", "modelo", "#dd8452")):
        v = mem[col].to_numpy() / 1024
        ax[1].barh(y, v, left=esq, color=cor, label=nome); esq += v
    ax[1].axvline(mem["flash_max_bytes"].iloc[0] / 1024, color="r", ls="--", label="capacidade")
    ax[1].set_yticks(y); ax[1].set_yticklabels(VAR_OK); ax[1].set_xlabel("KB"); ax[1].set_title("Flash")
    ax[1].legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3, frameon=False)
    fig.tight_layout(); salvar_fig(fig, "memoria"); plt.show()

In [ ]:
# Estimativa do notebook de treino x arena que o TFLM realmente usou
est_vs_real = pd.DataFrame()
if VAR_OK and meta.get("variants"):
    linhas = []
    for v in VAR_OK:
        e = meta["variants"].get(v)
        if e:
            real = mem.loc[v, "arena_used_bytes"] / 1024
            linhas.append({"variante": v, "estimativa_mín_KB": e["arena_min_KB"], "estimativa_máx_KB": e["arena_max_KB"],
                           "arena_real_KB": real, "real_vs_estimativa_máx_%": 100 * (real / e["arena_max_KB"] - 1)})
    est_vs_real = pd.DataFrame(linhas).set_index("variante")
    display(est_vs_real.round(2))
    md("A estimativa considera só os tensores de ativação. O arena real inclui alinhamento, buffers de *scratch* dos kernels e estruturas internas, "
       "então costuma ficar um pouco acima da estimativa máxima.")

if len(nao_exec):
    md("**Versões que não executaram**")
    linhas = []
    for v, r in nao_exec.iterrows():
        linhas.append({"variante": v, "situação": STATUS_TXT.get(r["status"], r["status"]),
                       "arena_máx_que_cabe_KB": r.get("arena_max_link_kb"), "arena_mín_necessário_KB": r.get("arena_min_est_kb"),
                       "modelo_KB": r.get("tflite_bytes", np.nan) / 1024})
    nf = pd.DataFrame(linhas).set_index("variante")
    nf["falta_KB"] = nf["arena_mín_necessário_KB"] - nf["arena_máx_que_cabe_KB"]
    display(nf.round(1))

## 5. Acurácia (inferência feita na placa)

O conjunto de teste é pequeno (uma janela vale cerca de 0,4 ponto percentual), então os intervalos de confiança abaixo são largos. Diferenças de poucas janelas entre versões não são evidência de que uma versão seja melhor.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = k / n; den = 1 + z ** 2 / n
    c = (p + z ** 2 / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return c - h, c + h


def matriz_confusao(y, p, k):
    m = np.zeros((k, k), dtype=int)
    for a, b in zip(y, p):
        m[int(a), int(b)] += 1
    return m


def metricas_classe(m):
    tp = np.diag(m).astype(float); fp = m.sum(0) - tp; fn = m.sum(1) - tp
    prec = np.divide(tp, tp + fp, out=np.zeros(len(tp)), where=(tp + fp) > 0)
    rec = np.divide(tp, tp + fn, out=np.zeros(len(tp)), where=(tp + fn) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(len(tp)), where=(prec + rec) > 0)
    return prec, rec, f1, m.sum(1)


acc_tab, cms, por_classe = pd.DataFrame(), {}, {}
VAR_ACC = []
if acc is not None and len(acc):
    VAR_ACC = [v for v in ORDEM if v in set(acc["variante"])]
    linhas = []
    for v in VAR_ACC:
        s = acc[acc["variante"] == v]
        m = matriz_confusao(s["y_true"], s["y_pred"], K); cms[v] = m
        prec, rec, f1, sup = metricas_classe(m)
        por_classe[v] = pd.DataFrame({"precisão": prec, "recall": rec, "F1": f1, "janelas": sup}, index=CLASSES)
        k_ok = int((s["y_true"] == s["y_pred"]).sum()); n = len(s)
        lo, hi = wilson(k_ok, n)
        ref = primeira.loc[v].get("acc_tflite_pc") if v in primeira.index else np.nan
        linhas.append({"variante": v, "janelas": n, "acurácia": k_ok / n, "IC95_inf": lo, "IC95_sup": hi, "F1_macro": f1.mean(),
                       "acurácia_TFLite_PC": ref, "Δ_placa_vs_PC_pp": 100 * (k_ok / n - ref) if pd.notna(ref) else np.nan,
                       "concordância_com_PC": primeira.loc[v].get("concordancia_tflite_pc") if v in primeira.index else np.nan})
    acc_tab = pd.DataFrame(linhas).set_index("variante")
    display(acc_tab.round(4))
    acc_tab.to_csv(TAB_DIR / "acuracia.csv", encoding="utf-8-sig")
    md("**Por classe**")
    for v in VAR_ACC:
        print(v); display(por_classe[v].round(3))
else:
    print("Sem dados de acurácia na placa (rode sem --skip-accuracy).")

In [ ]:
if VAR_ACC:
    fig, axs = plt.subplots(1, len(VAR_ACC), figsize=(4.6 * len(VAR_ACC), 4.2), squeeze=False)
    for a, v in zip(axs[0], VAR_ACC):
        m = cms[v]; norm = m / np.maximum(m.sum(1, keepdims=True), 1)
        im = a.imshow(norm, vmin=0, vmax=1, cmap="Blues")
        for i in range(K):
            for j in range(K):
                a.text(j, i, f"{m[i, j]}\n{100 * norm[i, j]:.0f}%", ha="center", va="center", fontsize=8,
                       color="white" if norm[i, j] > 0.55 else "black")
        a.set_xticks(range(K)); a.set_xticklabels(CLASSES, rotation=35, ha="right")
        a.set_yticks(range(K)); a.set_yticklabels(CLASSES)
        a.set_xlabel("previsto"); a.set_ylabel("real"); a.set_title(f"{v} (acurácia {acc_tab.loc[v, 'acurácia']:.1%})"); a.grid(False)
    fig.tight_layout(); salvar_fig(fig, "matrizes_confusao"); plt.show()

### Acordo entre versões e janelas difíceis

Como todas as versões viram **as mesmas janelas**, dá para comparar par a par. O teste de McNemar considera só as janelas em que uma versão acerta e a outra erra.

In [ ]:
mcn, acordo = pd.DataFrame(), pd.DataFrame()
dificeis = None
if len(VAR_ACC) >= 1:
    P = acc.pivot(index="janela", columns="variante", values="y_pred")[VAR_ACC]
    Y = acc.drop_duplicates("janela").set_index("janela")["y_true"].reindex(P.index)
    certo = P.eq(Y, axis=0)
    if len(VAR_ACC) > 1:
        acordo = pd.DataFrame({b: {a: float((P[a] == P[b]).mean()) for a in VAR_ACC} for b in VAR_ACC})
        md("**Concordância das predições entre versões**"); display(acordo.round(3))
        linhas = []
        for i, a in enumerate(VAR_ACC):
            for b in VAR_ACC[i + 1:]:
                so_a = int((certo[a] & ~certo[b]).sum()); so_b = int((~certo[a] & certo[b]).sum())
                p = stats.binomtest(min(so_a, so_b), so_a + so_b, 0.5).pvalue if so_a + so_b > 0 else 1.0
                linhas.append({"par": f"{a} × {b}", f"só a 1ª acerta": so_a, "só a 2ª acerta": so_b,
                               "predições diferentes": int((P[a] != P[b]).sum()), "p_McNemar": p})
        mcn = pd.DataFrame(linhas).set_index("par")
        md("**McNemar**"); display(mcn.round(4))
    erra_todas = ~certo.any(axis=1)
    dificeis = Y[erra_todas]
    md(f"**Janelas que todas as versões erram:** {int(erra_todas.sum())} de {len(P)} ({100 * erra_todas.mean():.1f}%). "
       + ("Por classe real: " + ", ".join(f"{CLASSES[int(c)]}: {n}" for c, n in dificeis.value_counts().sort_index().items()) if len(dificeis) else ""))

    # confiança (maior score) nas janelas certas x erradas
    sc = [c for c in acc.columns if c.startswith("score_")]
    if sc:
        linhas = []
        for v in VAR_ACC:
            s = acc[acc["variante"] == v]; conf = s[sc].max(axis=1); ok_ = s["y_true"] == s["y_pred"]
            linhas.append({"variante": v, "confiança_média_acertos": conf[ok_].mean(),
                           "confiança_média_erros": conf[~ok_].mean() if (~ok_).any() else np.nan})
        display(pd.DataFrame(linhas).set_index("variante").round(3))

## 6. Trade-offs

In [ ]:
trade = pd.DataFrame()
if len(VAR_OK) >= 1 and not estat.empty:
    base_v = next(v for v in ORDEM if v in VAR_OK and v in estat.index)      # referência: a versão de maior precisão que executou
    trade = pd.DataFrame(index=[v for v in VAR_OK if v in estat.index])
    trade["latência_ms"] = estat["média_ms"]
    trade["RAM_KB"] = mem.loc[trade.index, "ram_bytes"] / 1024
    trade["arena_KB"] = mem.loc[trade.index, "arena_used_bytes"] / 1024
    trade["modelo_KB"] = mem.loc[trade.index, "tflite_bytes"] / 1024
    if len(acc_tab):
        trade["acurácia"] = acc_tab["acurácia"]; trade["F1_macro"] = acc_tab["F1_macro"]
    rel = pd.DataFrame(index=trade.index)
    rel[f"speedup_vs_{base_v}"] = trade.loc[base_v, "latência_ms"] / trade["latência_ms"]
    rel[f"arena_vs_{base_v}"] = trade["arena_KB"] / trade.loc[base_v, "arena_KB"]
    rel[f"modelo_vs_{base_v}"] = trade["modelo_KB"] / trade.loc[base_v, "modelo_KB"]
    if "F1_macro" in trade:
        rel[f"ΔF1_vs_{base_v}_pp"] = 100 * (trade["F1_macro"] - trade.loc[base_v, "F1_macro"])
    md(f"**Relativo a {base_v}** (a versão de maior precisão que executou na placa)")
    display(rel.round(3))
    trade.to_csv(TAB_DIR / "tradeoffs.csv", encoding="utf-8-sig")

    pareto = []
    if "F1_macro" in trade and len(trade) > 1:
        for a in trade.index:
            dominada = False
            for b in trade.index:
                if b == a:
                    continue
                nao_pior = (trade.loc[b, "latência_ms"] <= trade.loc[a, "latência_ms"] and trade.loc[b, "RAM_KB"] <= trade.loc[a, "RAM_KB"]
                            and trade.loc[b, "F1_macro"] >= trade.loc[a, "F1_macro"])
                melhor = (trade.loc[b, "latência_ms"] < trade.loc[a, "latência_ms"] or trade.loc[b, "RAM_KB"] < trade.loc[a, "RAM_KB"]
                          or trade.loc[b, "F1_macro"] > trade.loc[a, "F1_macro"])
                if nao_pior and melhor:
                    dominada = True
            if not dominada:
                pareto.append(a)
        md(f"**Fronteira de Pareto** (menor latência, menor RAM, maior F1): {', '.join(pareto)}")

    if "F1_macro" in trade:
        fig, ax = plt.subplots(figsize=(6.4, 4.2))
        for v in trade.index:
            ax.errorbar(trade.loc[v, "latência_ms"], 100 * trade.loc[v, "F1_macro"], xerr=estat.loc[v, "dp_ms"], fmt="none", ecolor="gray", capsize=3)
            ax.scatter(trade.loc[v, "latência_ms"], 100 * trade.loc[v, "F1_macro"], s=trade.loc[v, "RAM_KB"] * 4, color=CORES[v], alpha=0.75)
            ax.annotate(v, (trade.loc[v, "latência_ms"], 100 * trade.loc[v, "F1_macro"]), textcoords="offset points", xytext=(8, 8))
        ax.set_xlabel("latência (ms)"); ax.set_ylabel("F1-macro (%)"); ax.set_title("Latência x F1 (tamanho da bolha = RAM estática)")
        fig.tight_layout(); salvar_fig(fig, "tradeoff_latencia_f1"); plt.show()

## 7. Insights automáticos

Resumo gerado a partir dos números acima. Serve como ponto de partida para a discussão, e os textos devem ser conferidos contra as tabelas antes de ir para o relatório.

In [ ]:
ins = []
n_sessoes = sessoes["sessao"].nunique()

# ---- execução ----
if VAR_OK:
    ins.append(f"**Execução.** Versões medidas na placa: {', '.join(VAR_OK)} ({n_sessoes} sessão(ões)).")
for v, r in nao_exec.iterrows():
    if r["status"] == "no_fit_ram" and pd.notna(r.get("arena_max_link_kb")):
        ins.append(f"**{v} não executa no Nano 33 BLE.** Depois da RAM usada pelo sistema, cabem no máximo ~{r['arena_max_link_kb']:.0f} KB "
                   f"de arena, e o modelo precisa de pelo menos ~{r['arena_min_est_kb']:.0f} KB (faltam ~{r['arena_min_est_kb'] - r['arena_max_link_kb']:.0f} KB). "
                   f"Isso é um resultado do benchmark, e não uma falha do experimento.")
    else:
        ins.append(f"**{v} não executou** ({STATUS_TXT.get(r['status'], r['status'])}).")

# ---- latência ----
if not estat.empty:
    ins.append("**Latência do Invoke().** " + "; ".join(f"{v}: {estat.loc[v, 'média_ms']:.2f} ± {estat.loc[v, 'dp_ms']:.2f} ms" for v in estat.index) + ".")
    for nome, r in comp.iterrows():
        a, b = nome.split(" → ")
        if r["IC95_inf"] > 1:
            ins.append(f"{b} é **{r['razão_médias']:.2f}× mais rápida** que {a} (IC95% {r['IC95_inf']:.2f} a {r['IC95_sup']:.2f}).")
        elif r["IC95_sup"] < 1:
            ins.append(f"{b} é **{1 / r['razão_médias']:.2f}× mais lenta** que {a} (IC95% da razão {r['IC95_inf']:.2f} a {r['IC95_sup']:.2f}).")
        else:
            ins.append(f"{a} e {b} **não se distinguem** na latência (IC95% da razão {r['IC95_inf']:.2f} a {r['IC95_sup']:.2f}).")
    if len(qual):
        pior_cv = estat["cv_%"].idxmax()
        ins.append(f"**Estabilidade.** O coeficiente de variação vai de {estat['cv_%'].min():.2f}% a {estat['cv_%'].max():.2f}% (maior em {pior_cv}); "
                   f"picos acima de 5 MAD: até {qual['picos_>5MAD_%'].max():.1f}% das execuções.")
        for v in qual.index:
            if pd.notna(qual.loc[v, "p_deriva"]) and qual.loc[v, "p_deriva"] < 0.01 and abs(qual.loc[v, "deriva_%_por_100_exec"]) > 1:
                ins.append(f"Atenção: {v} tem deriva de {qual.loc[v, 'deriva_%_por_100_exec']:+.2f}% por 100 execuções (p = {qual.loc[v, 'p_deriva']:.3g}). "
                           f"Confira a série temporal (aquecimento, alimentação USB).")
        if qual["p_sessões_kruskal"].notna().any() and (qual["p_sessões_kruskal"] < 0.01).any():
            v = qual["variação_entre_sessões_%"].idxmax()
            ins.append(f"As sessões diferem entre si (até {qual['variação_entre_sessões_%'].max():.1f}% em {v}). "
                       f"Reporte a variação entre sessões, e não só o desvio padrão dentro delas.")
    if len(indep) and indep["p_diferença_entre_classes"].notna().any():
        if (indep["p_diferença_entre_classes"] < 0.01).any():
            ins.append("A latência **depende da classe** do sinal em alguma versão (p < 0,01). Repetir uma única janela pode não representar o uso real.")
        else:
            ins.append("A latência **não depende do conteúdo** do sinal (Kruskal entre classes sem diferença), então repetir uma janela representa bem o uso real.")
    elif len(indep):
        ins.append("Não foi possível testar se a latência depende do conteúdo do sinal (janelas processadas em ordem). Reexecute com o `run_benchmark.py` atual.")

# ---- tempo real ----
if len(rt):
    ok_rt = [v for v in rt.index if rt.loc[v, "uso_do_intervalo_%"] <= 100]
    ins.append("**Tempo real.** " + "; ".join(f"{v} ocupa {rt.loc[v, 'uso_do_intervalo_%']:.1f}% do intervalo entre janelas" for v in rt.index)
               + (" (todas acompanham o sensor)." if len(ok_rt) == len(rt) else f". Só {', '.join(ok_rt) or 'nenhuma'} acompanha(m) o sensor."))

# ---- memória ----
if len(mem):
    ins.append("**Memória.** " + "; ".join(f"{v}: RAM {100 * mem.loc[v, 'ram_bytes'] / mem.loc[v, 'ram_max_bytes']:.1f}% (arena {mem.loc[v, 'arena_used_bytes'] / 1024:.0f} KB), "
                                            f"Flash {100 * mem.loc[v, 'flash_bytes'] / mem.loc[v, 'flash_max_bytes']:.1f}% (modelo {mem.loc[v, 'tflite_bytes'] / 1024:.0f} KB)"
                                            for v in mem.index) + ".")
    fl = mem["flash_runtime_bytes"] / mem["flash_bytes"]
    ins.append(f"O modelo é só {100 * (1 - fl).min():.1f}% a {100 * (1 - fl).max():.1f}% da Flash usada. O resto é o sistema e o TFLM, então a Flash quase não "
               f"diferencia as versões. O que diferencia é a **RAM (arena)**.")

# ---- acurácia ----
if len(acc_tab):
    ins.append("**Acurácia na placa.** " + "; ".join(f"{v}: {acc_tab.loc[v, 'acurácia']:.1%} (IC95% {acc_tab.loc[v, 'IC95_inf']:.1%} a {acc_tab.loc[v, 'IC95_sup']:.1%}, F1-macro {acc_tab.loc[v, 'F1_macro']:.3f})" for v in acc_tab.index) + ".")
    if len(acc_tab) > 1:
        faixa = acc_tab["acurácia"].max() - acc_tab["acurácia"].min()
        sobrepoe = acc_tab["IC95_inf"].max() <= acc_tab["IC95_sup"].min()
        ins.append(f"A diferença entre a melhor e a pior versão é de {100 * faixa:.1f} p.p. "
                   + ("Os intervalos de confiança se sobrepõem, então **com este conjunto de teste a quantização não mostra perda mensurável de acurácia**."
                      if sobrepoe else "Os intervalos de confiança não se sobrepõem: a diferença pode ser real. Confira o McNemar."))
        if len(mcn) and (mcn["p_McNemar"] < 0.05).any():
            ins.append("McNemar significativo (p < 0,05) em: " + ", ".join(mcn.index[mcn["p_McNemar"] < 0.05]) + ".")
    if acc_tab["concordância_com_PC"].notna().any():
        conc = acc_tab["concordância_com_PC"].dropna()
        if (conc < 0.999).any():
            ins.append("**Alerta:** a placa não reproduz exatamente as predições do TFLite do PC em " + ", ".join(f"{v} ({conc[v]:.1%})" for v in conc.index if conc[v] < 0.999) +
                       ". Vale investigar kernels ou quantização.")
        else:
            ins.append("A placa reproduz **100% das predições** do TFLite do PC em todas as versões medidas.")
    if dificeis is not None and len(dificeis):
        ins.append(f"{len(dificeis)} janela(s) são erradas por todas as versões. Provavelmente é dificuldade do problema/split, e não efeito da quantização.")

# ---- trade-off ----
if len(trade):
    if len(rel.columns) and len(trade) > 1:
        c_sp = [c for c in rel.columns if c.startswith("speedup")][0]; c_ar = [c for c in rel.columns if c.startswith("arena_vs")][0]
        for v in rel.index:
            if v != base_v:
                ins.append(f"**{v} vs {base_v}:** {rel.loc[v, c_sp]:.2f}× a velocidade, {rel.loc[v, c_ar]:.2f}× o arena"
                           + (f", ΔF1 = {rel.loc[v, [c for c in rel.columns if c.startswith('ΔF1')][0]]:+.1f} p.p" if any(c.startswith('ΔF1') for c in rel.columns) else "") + ".")
    if "pareto" in globals() and pareto:
        ins.append(f"Fronteira de Pareto (latência, RAM, F1): **{', '.join(pareto)}**.")

# ---- ressalvas ----
if n_sessoes < 3:
    ins.append(f"**Ressalva:** só {n_sessoes} sessão(ões). O planejamento prevê sessões com ordem aleatória, e com poucas sessões não dá para separar efeito da versão de efeito da ordem.")
if lat is not None and len(lat) and (lat.groupby(["variante", "sessao"]).size() < 1000).any():
    ins.append("Algumas sessões têm menos de 1000 execuções, abaixo do previsto no canvas.")
ins.append("Energia e EDP não entram nesta análise (sem medidor).")

md("\n".join(f"- {t}" for t in ins))
(TAB_DIR / "insights.md").write_text("\n".join(f"- {t}" for t in ins), encoding="utf-8")

## 8. Energia e EDP (quando houver medida)

Quando tiver a energia por inferência (µJ) medida com o INA219/INA226, preencha o dicionário abaixo e rode a célula. Ela calcula o **EDP** (energia × tempo), a potência média durante o `Invoke()` e a potência média se a placa processar todas as janelas continuamente.

In [ ]:
ENERGIA_UJ = {}   # exemplo: {"INT8": 135.0, "INT16": 218.0}   (µJ por inferência)

if ENERGIA_UJ and not estat.empty:
    hop = 1000 * WINDOW_LEN / FS_HZ * (1 - OVERLAP)
    e = pd.DataFrame({"energia_uJ": pd.Series(ENERGIA_UJ)}).reindex(estat.index).dropna()
    e["latência_ms"] = estat["média_ms"]
    e["EDP_uJ_ms"] = e["energia_uJ"] * e["latência_ms"]
    e["potência_no_Invoke_mW"] = e["energia_uJ"] / e["latência_ms"]               # µJ / ms = mW
    e["potência_média_contínua_mW"] = e["energia_uJ"] * (1000 / hop) / 1000       # uma inferência por janela
    display(e.round(3))
    e.to_csv(TAB_DIR / "energia_edp.csv", encoding="utf-8-sig")
else:
    md("Preencha `ENERGIA_UJ` para calcular o EDP.")

## 9. Arquivos gerados

In [ ]:
for pasta in (FIG_DIR, TAB_DIR):
    print(pasta.resolve())
    for f in sorted(pasta.iterdir()):
        print(f"   {f.name}")